## Feature Engineering
---
### Objectives

- Select meaningful features for the modeling

- Remove noise and irrelavant colomns

- Encoding Categorical columns

- Making dataset in numeric format so it would be final for the machine learning

### Loading Cleaned Dataset

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/medical_appointment_no_show_cleaned.csv')
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,Handcap_binary,WaitingDays
0,9.598513e+13,5626772,F,2016-04-27 08:36:51+00:00,2016-04-29 00:00:00+00:00,76,REPÚBLICA,0,1,0,0,0,0,No,0,1
1,7.336882e+14,5630279,F,2016-04-27 15:05:12+00:00,2016-04-29 00:00:00+00:00,23,GOIABEIRAS,0,0,0,0,0,0,Yes,0,1
2,3.449833e+12,5630575,F,2016-04-27 15:39:58+00:00,2016-04-29 00:00:00+00:00,39,GOIABEIRAS,0,0,0,0,0,0,Yes,0,1
3,7.812456e+13,5629123,F,2016-04-27 12:48:25+00:00,2016-04-29 00:00:00+00:00,19,CONQUISTA,0,0,0,0,0,0,No,0,1
4,7.345362e+14,5630213,F,2016-04-27 14:58:11+00:00,2016-04-29 00:00:00+00:00,30,NOVA PALESTINA,0,0,0,0,0,0,No,0,1


### Dropping Columns
---
- Features like `PatientId` `AppointmentID` `ScheduledDay` `AppointmentDay` are not useful in modelling and will only add noise in the dataset.

- Correct way to drop these columns so only relavant features can be preserved.

- `Waiting Days` feature is already extracted from the dates so it is kept and other date coloumns are dropped.

- `Handcap_binary` is also generated from `Handcap` feature so it can also be dropped.

In [3]:
df = df.drop(['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay','Handcap','Neighbourhood'], axis = 1)
df.head()

,Gender,Age,Scholarship,Hipertension,Diabetes,Alcoholism,SMS_received,No-show,Handcap_binary,WaitingDays
0,F,76,0,1,0,0,0,No,0,1
1,F,23,0,0,0,0,0,Yes,0,1
2,F,39,0,0,0,0,0,Yes,0,1
3,F,19,0,0,0,0,0,No,0,1
4,F,30,0,0,0,0,0,No,0,1


### Target Variable Analysis
---
- The goal is to confirm that target variable values are in string or numeric format.

- String values like `yes` or `no` are not ideal for machine learning model so conversion into numeric format is neccessary.

In [4]:
df['No-show'].value_counts()

No-show
No     49894
Yes    19934
Name: count, dtype: int64

### Conversion Needed
---
- As target variable values are in string format which cannot be understand by the model conversion into numerical values is needed.

In [5]:
df['No-show'] = df['No-show'].map({'No':0, 'Yes':1})
df['No-show'].value_counts()

No-show
0    49894
1    19934
Name: count, dtype: int64

### Handling Categorical column
---
- Encoding of category feature `Gender` is also neccessary in order make the whole dataset in numeric format

In [6]:
df['Gender'].unique()

array(['F', 'M'], dtype=object)

### Binary Encoding (Gender)
---
- As we have two values binary encoding is the most suitable technique which can be applied.

- `F` will be represented as `0` and `M` will be represented as `1`.

In [7]:
df['Gender'] = df['Gender'].map({'F':0, 'M':1})
df['Gender'].value_counts()

Gender
0    47030
1    22798
Name: count, dtype: int64

In [8]:
df.shape

(69828, 10)

### Dataset Shape
---
- After dropping columns now dataset has only `10` features which are important for the modeling.

### Seperating Features
---
- The goal is to seperate target feature from the other features.

- Seperation is needed so that relevant features only can be used as input for the model.

- It will be used during the train-test-split

In [9]:
x = df.drop('No-show',axis = 1)
y = df['No-show']

### Splitting Dataset
---
- `train-test-split` is introduced to split the data set into training as well as testing data.

- Proportion will be **80-20** and stratify is also used to ensure the equal distribution of target variable.

In [10]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

### Scaling
---
- In order to ensure that no numerical data influence the model with its outlying value the scaling is applied to data.


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

### Importing and Training Model
---
- `Logistic Regression` as a baseline model is imported.

- The goal is to train the model on the whole training dataset so it can detect patterns and make accurate predictions during evaluation.

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)

### Evaluation of Model
---
- The goal is to evaluate the predictions of the model based on some metrics.

- `Accuracy Score` `Confusion Matrix` `Classification Report` are the metrics used for evaluation.

- This will help to understand how model is performing and what changes can be applied if required.

In [13]:
from sklearn.metrics import accuracy_score,confusion_matrix, classification_report

print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('\n Confusion Matrix :\n',confusion_matrix(y_test, y_pred))
print('\n Classification Report :\n',classification_report(y_test, y_pred))

Accuracy Score : 0.5559215237004153

 Confusion Matrix :
 [[5499 4480]
 [1722 2265]]

 Classification Report :
               precision    recall  f1-score   support

           0       0.76      0.55      0.64      9979
           1       0.34      0.57      0.42      3987

    accuracy                           0.56     13966
   macro avg       0.55      0.56      0.53     13966
weighted avg       0.64      0.56      0.58     13966



### Changing Threshold 
---
 Goal is to introduce effecient balance between `accuracy` `precision` and `recall`.

In [14]:
y_prob = model.predict_proba(x_test_scaled)[:, 1]
threshold = 0.45

y_pred_new = (y_prob > threshold).astype(int)

print('Accuracy Score :',accuracy_score(y_test, y_pred_new))
print('\n Confusion Matrix :\n',confusion_matrix(y_test, y_pred_new))
print('\n Classification Report :\n',classification_report(y_test, y_pred_new))

Accuracy Score : 0.46577402262637835

 Confusion Matrix :
 [[3366 6613]
 [ 848 3139]]

 Classification Report :
               precision    recall  f1-score   support

           0       0.80      0.34      0.47      9979
           1       0.32      0.79      0.46      3987

    accuracy                           0.47     13966
   macro avg       0.56      0.56      0.47     13966
weighted avg       0.66      0.47      0.47     13966



### Threshold Tuning Insight
---
I experimented with probability thresholds to improve recall.

- At threshold 0.5 → Recall = 57%
- At threshold 0.45 → Recall improved to 79%
- At threshold 0.4 → Recall = 93% but too many false positives

Threshold 0.45 provides a better balance between recall and precision.

In [15]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)
print('Auc Score : ',auc)

Auc Score :  0.5883545940581064


###  Evaluation Observations
---
- The overall accuracy of the model is approximately **56%**, which is relatively low. 
  However, since the dataset is imbalanced, accuracy alone is not a reliable performance indicator.

- The confusion matrix shows around **1,700 False Negatives (FN)**.  
  This means a significant number of actually risky (no-show) patients were predicted as non-risky.  
  In a healthcare context, this is concerning because these patients would not receive preventive intervention.

- The model achieves a **recall of 57% for risky patients (class 1)**.  
  This indicates that the model is able to correctly detect more than half of the actual no-show cases, which is important in risk identification scenarios.

- However, the **precision for risky patients is only 34%**, meaning many patients flagged as risky are actually not no-shows.  
  This could lead to unnecessary follow-ups or resource allocation.

- The **AUC score of 0.588** suggests that the model has weak but better-than-random class separation ability.  
  It indicates that the model struggles to clearly distinguish between no-show and non–no-show patients.


  ---

### Threshold Tuning Insight

To improve risky patient detection, probability threshold tuning was performed:

- At threshold 0.45 → Recall improved to **79%**
- False Negatives reduced significantly
- Precision slightly decreased

This demonstrates that adjusting the decision threshold allows better alignment with business objectives, especially when prioritizing recall over accuracy.

### Random Forest Classifier
---


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    class_weight='balanced'
)

rf_model.fit(x_train_scaled,y_train)
y_pred_rf = rf_model.predict(x_test_scaled)

print('Accuracy : ',accuracy_score(y_test, y_pred_rf))
print('\nConfusion Matrix :\n',confusion_matrix(y_test,y_pred_rf))
print('\nClassification Report :\n', classification_report(y_test, y_pred_rf))

Accuracy :  0.6000286409852499

Confusion Matrix :
 [[6941 3038]
 [2548 1439]]

Classification Report :
               precision    recall  f1-score   support

           0       0.73      0.70      0.71      9979
           1       0.32      0.36      0.34      3987

    accuracy                           0.60     13966
   macro avg       0.53      0.53      0.53     13966
weighted avg       0.61      0.60      0.61     13966



In [17]:
y_prob_rf = rf_model.predict_proba(x_test_scaled)[:, 1]

auc_rf = roc_auc_score(y_test, y_prob_rf)
print("AUC score : ", auc_rf)

AUC score :  0.5406857410343512


### Model Comparison Summary
---
I compared Logistic Regression and Random Forest models.

- Logistic Regression achieved higher recall for risky patients.
- Logistic Regression also showed better AUC (≈0.59).
- Random Forest slightly improved accuracy but reduced recall significantly.
- Since missing risky patients is more critical, Logistic Regression was selected as the better model.

Conclusion:
Model selection was based on recall and AUC rather than accuracy alone.

In [18]:
df.to_csv('../data/processed/final_model_dataset.csv',index=False)